# Qwen Family Testing on Kaggle
This notebook is specifically tailored to test the Qwen Family models.

In [ ]:
"""
================================================================================
KAGGLE CELL 1: Environment Setup & Installations
Run this cell first to install all necessary packages for the Qwen models.
================================================================================
"""
import torch, torchvision
with open("constraints.txt", "w") as f:
    f.write(f"torch=={torch.__version__}\n")
    f.write(f"torchvision=={torchvision.__version__}\n")

!pip install -c constraints.txt transformers accelerate bitsandbytes qwen-vl-utils --upgrade
# Align Pillow after transformers to fix _Ink version mismatch
!pip install "Pillow>=11.2.1" --upgrade -q
# Remove torchaudio to avoid CUDA version mismatch (not needed for vision)
!pip uninstall torchaudio -y -q 2>/dev/null || true


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Runtime compatibility shims - MUST run before 'import torch'
# ══════════════════════════════════════════════════════════════════════
import sys, importlib

# Fix 1: Pillow _Ink ImportError (torchvision <-> Pillow version mismatch)
try:
    _pil_typing = importlib.import_module("PIL._typing")
    if not hasattr(_pil_typing, "_Ink"):
        _pil_typing._Ink = str | float | tuple[int, ...]
        print("[shim] Patched PIL._typing._Ink")
except ImportError:
    pass

# Fix 2: Block torchaudio to prevent CUDA version mismatch crash
# (transformers.loss.loss_rnnt imports torchaudio; we don't need it for vision)
try:
    import torchaudio as _ta_test
except Exception:
    # torchaudio is broken or missing - block it so transformers skips it
    sys.modules["torchaudio"] = None
    print("[shim] Blocked broken torchaudio (not needed for vision models)")

import torch
import gc
from PIL import Image
import glob
import os

# A helper function to prevent Kaggle from crashing due to out-of-memory (OOM) errors.
def clear_vram():
    """Wipes the GPU memory cleanly between model tests."""
    gc.collect()
    torch.cuda.empty_cache()
    print("VRAM Cleared.")

print("Searching for images in /kaggle/input/ and /kaggle/working/ ...")
all_files = glob.glob("/kaggle/input/**/*.*", recursive=True) + glob.glob("/kaggle/working/**/*.*", recursive=True)
TEST_IMAGE_PATHS = sorted([
    p for p in all_files 
    if p.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))
])[:10]

if not TEST_IMAGE_PATHS:
    print("WARNING: Could not find any images! Downloading fallbacks...")
    import urllib.request
    os.makedirs("/kaggle/working/samples", exist_ok=True)
    urls = [
        "https://raw.githubusercontent.com/QwenLM/Qwen-VL/master/assets/mm_tutorial/Doc.jpg",
        "https://raw.githubusercontent.com/QwenLM/Qwen-VL/master/assets/mm_tutorial/Receipt.jpg"
    ]
    for i, url in enumerate(urls):
        path = f"/kaggle/working/samples/sample_{i}.jpg"
        urllib.request.urlretrieve(url, path)
        TEST_IMAGE_PATHS.append(path)

print(f"Loaded {len(TEST_IMAGE_PATHS)} images for testing:")
for p in TEST_IMAGE_PATHS:
    print(f" - {p}")

TEST_PROMPT = "Extract all handwritten text verbatim and format it."


In [ ]:
"""
================================================================================
KAGGLE CELL 2: The Qwen Series testing function
Alibaba's SOTA models. We load them in 4-bit/8-bit to fit the Kaggle T4.
================================================================================
"""
def test_qwen_family(image_paths, model_id="Qwen/Qwen2.5-VL-3B-Instruct"):
    print(f"\n--- Loading {model_id} ---")
    
    # T4 GPUs only support float16, not bfloat16!
    load_kwargs = {"device_map": "auto", "torch_dtype": torch.float16, "attn_implementation": "eager"}
    
    # Check if the model is large (7B or 8B parameter). 
    # Note: We must exclude '0.8B' models from matching '8B'
    is_large_model = "7B" in model_id or ("8B" in model_id and "0.8B" not in model_id)
    
    if is_large_model:
        from transformers import BitsAndBytesConfig
        # Use BitsAndBytesConfig instead of deprecated load_in_4bit kwarg to avoid remote code errors
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        load_kwargs["quantization_config"] = quantization_config

    is_vl = "VL" in model_id

    if is_vl:
        if "2.5" in model_id:
            from transformers import Qwen2_5_VLForConditionalGeneration as ModelClass
        elif "3" in model_id:
            try:
                from transformers import Qwen3VLForConditionalGeneration as ModelClass
            except ImportError:
                from transformers import AutoModelForImageTextToText as ModelClass
        else:
            from transformers import Qwen2VLForConditionalGeneration as ModelClass
    else:
        from transformers import AutoModelForCausalLM as ModelClass

    model = ModelClass.from_pretrained(model_id, trust_remote_code=True, **load_kwargs)
    
    if is_vl:
        from transformers import AutoProcessor
        from qwen_vl_utils import process_vision_info
        processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        for i, img_path in enumerate(image_paths):
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": img_path},
                        {"type": "text", "text": TEST_PROMPT},
                    ],
                }
            ]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = processor(
                text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
            )
            inputs = {k: v.to(torch.float16).to("cuda") if v.is_floating_point() else v.to("cuda") for k, v in inputs.items()}
            generated_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)
            generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)]
            output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
            print(f"{model_id} Output for Image {i+1}:\n", output_text[0])
            del inputs, generated_ids, generated_ids_trimmed, output_text
            torch.cuda.empty_cache()
    else:
        from transformers import AutoTokenizer
        print(f"Warning: {model_id} is a text-only model! Passing prompt without image.")
        processor = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        messages = [{"role": "user", "content": TEST_PROMPT}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor([text], return_tensors="pt", padding=True).to("cuda")
        generated_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)]
        output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        print(f"{model_id} Output:\n", output_text[0])
        del inputs, generated_ids, generated_ids_trimmed, output_text
        torch.cuda.empty_cache()

    del model, processor
    clear_vram()


In [ ]:
test_qwen_family(TEST_IMAGE_PATHS, "Qwen/Qwen2.5-VL-3B-Instruct")

In [ ]:
test_qwen_family(TEST_IMAGE_PATHS, "Qwen/Qwen2.5-VL-7B-Instruct")

In [ ]:
test_qwen_family(TEST_IMAGE_PATHS, "Qwen/Qwen2-VL-2B-Instruct")